In [ ]:
import pandas as pd
from dotenv import load_dotenv
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

load_dotenv()
ACTIVE_BACKEND = "GPU"
TEST_DOC_LIMIT = 100 # Set to None to use all documents, or a positive integer for testing with a subset
MAX_NEW_TOKENS = 50 # Adjust based on expected label length and model capabilities
PARQUET_PATH = r"C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260310_180458_ADS_Curation_Run\data\publications_disambiguated.parquet"

df = pd.read_parquet(PARQUET_PATH)
docs = df["full_text"].dropna().astype(str).str.strip()
docs = docs[docs != ""].tolist()[:TEST_DOC_LIMIT] if TEST_DOC_LIMIT else docs[docs != ""].tolist()
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# `embeddings.position_ids | UNEXPECTED` is benign for this MiniLM checkpoint
# as long as no additional missing or unexpected weights are reported.

PHYSICS_PROMPT = (
    "You are an experienced researcher in gravitational physics, astrophysics, and cosmology. "
    "You are labeling research topic clusters based on scientific abstracts.\n\n"
    "Documents: [DOCUMENTS]\n"
    "Keywords: [KEYWORDS]\n\n"
    "Task:\n"
    "- Generate EXACTLY ONE topic label.\n"
    "- The label MUST contain between 4 and 7 words (inclusive).\n"
    "- The label should read like a review article or textbook chapter title.\n\n"
    "Guidelines:\n"
    "- Use standard scientific terminology from the field.\n"
    "- Be specific about the physical phenomenon or method.\n"
    "- Avoid generic terms like 'studies', 'analysis', or 'research'.\n\n"
    "Output format (single line):\n"
    "topic: <label>\n\n"
    "Do NOT write anything else."
)

QWEN_CHAT_PROMPT = (
    "<|im_start|>system\n"
    "You are an experienced researcher in gravitational physics, astrophysics, and cosmology. "
    "Return exactly one topic label in the format topic: <label>.\n"
    "<|im_end|>\n"
    "<|im_start|>user\n"
    f"{PHYSICS_PROMPT}\n"
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
)
QWEN_STOP = ["<|im_end|>"]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
if ACTIVE_BACKEND == "API":
    from bertopic.representation import LiteLLM

    representation_model = LiteLLM(
        model="openrouter/google/gemini-3.1-flash-lite-preview",
        prompt=PHYSICS_PROMPT,
        generator_kwargs={"max_tokens": MAX_NEW_TOKENS, "temperature": 0.0, "stop": ["\n"]},
    )

elif ACTIVE_BACKEND == "CPU":
    from bertopic.representation import LlamaCPP
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    model_path = hf_hub_download(
        repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        filename="qwen2.5-0.5b-instruct-q4_k_m.gguf",
    )
    llm = Llama(model_path=model_path, n_gpu_layers=0, n_ctx=16384, stop=QWEN_STOP, verbose=False)
    representation_model = LlamaCPP(
        llm,
        prompt=QWEN_CHAT_PROMPT,
        pipeline_kwargs={"max_tokens": MAX_NEW_TOKENS},
    )

elif ACTIVE_BACKEND == "GPU":
    from bertopic.representation import TextGeneration
    from transformers import pipeline

    try:
        generator = pipeline(
            "text-generation",
            model="Qwen/Qwen2.5-0.5B-Instruct",
            device_map="auto",
            dtype="auto",
        )
    except TypeError:
        generator = pipeline(
            "text-generation",
            model="Qwen/Qwen2.5-0.5B-Instruct",
            device_map="auto",
            torch_dtype="auto",
        )
    generation_config = getattr(getattr(generator, "model", None), "generation_config", None)
    if generation_config is not None:
        for attr in ("max_length", "temperature", "top_p", "top_k"):
            if hasattr(generation_config, attr):
                setattr(generation_config, attr, None)
    representation_model = TextGeneration(
        generator,
        prompt=QWEN_CHAT_PROMPT,
        pipeline_kwargs={"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "num_return_sequences": 1},
    )

else:
    raise ValueError(f"Invalid ACTIVE_BACKEND: {ACTIVE_BACKEND}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    representation_model=representation_model,
    verbose=False,
)
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info().head(10)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have bee

,Topic,Count,Name,Representation,Representative_Docs
0,-1,51,-1_topic: General Relativity and Quantum Gravi...,[topic: General Relativity and Quantum Gravity...,[The meaning of quantum gravity.. Should the g...
1,0,37,0_topic: General Relativity and Quantum Field ...,[topic: General Relativity and Quantum Field T...,[Gravitational field with zero points of the d...
2,1,12,1_topic: Cosmology and Gravitational Dynamics___,"[topic: Cosmology and Gravitational Dynamics, ...",[Boltzmann's cosmogony and the hierarchical st...
